# Sample stomach endoscope video

Stitches a short run of real frames from one already-verified-clean
real-camera trajectory (`Cameras/HighCam/Stomach-III/TumorfreeTrajectory_3` --
confirmed in PROGRESS.md's "Pose format" notes to have frame count == pose
row count exactly, 749/749, no dropped frames) into an actual `.mp4`, purely
so there's a genuine real-endoscope **stomach** video to test
`src/inference/reconstruct_video.py` against -- correct anatomy, correct
domain, same dataset this whole project already relies on.

No repo clone, no model, no requirements install needed here -- just
`opencv-python-headless`, already on Kaggle's base image. GPU off.

## 0. Resolve dataset root

In [ ]:
import os

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an endoslam dir under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

## 1. List frames from the chosen trajectory

In [ ]:
import glob

TRAJ_DIR = os.path.join(DATA_ROOT, "Cameras", "HighCam", "Stomach-III", "TumorfreeTrajectory_3")
frame_paths = sorted(
    glob.glob(os.path.join(TRAJ_DIR, "Frames", "*.jpg"))
    + glob.glob(os.path.join(TRAJ_DIR, "Frames", "*.png"))
)
assert frame_paths, f"no frames found under {TRAJ_DIR}/Frames"
print(f"{len(frame_paths)} total frames in this trajectory")

N_FRAMES = min(200, len(frame_paths))
selected = frame_paths[:N_FRAMES]
print(f"using the first {len(selected)} frames")

## 2. Stitch into an .mp4

In [ ]:
import cv2

FPS = 15
OUTPUT_PATH = "/kaggle/working/sample_stomach_endoscopy.mp4"

first = cv2.imread(selected[0])
H, W = first.shape[:2]
print(f"frame size: {W}x{H}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (W, H))
assert writer.isOpened(), "cv2.VideoWriter failed to open -- codec issue?"

for p in selected:
    frame = cv2.imread(p)
    writer.write(frame)
writer.release()

print(f"wrote {len(selected)} frames to {OUTPUT_PATH}")

## 3. Sanity check: re-open what we just wrote

In [ ]:
cap = cv2.VideoCapture(OUTPUT_PATH)
print("opened:", cap.isOpened())
count = 0
while True:
    ok, _ = cap.read()
    if not ok:
        break
    count += 1
cap.release()
print(f"frames readable back: {count}")
assert count == len(selected), f"expected {len(selected)} frames, read back {count}"
print("OK -- sample_stomach_endoscopy.mp4 is ready to download")